# Gold Layer: Review Themes

Produces one row per neighbourhood summarising the dominant review themes
derived from keyword matching across guest comments.

Routes automatically: if `BRISTOL_REVIEWS_SILVER` is confirmed it computes
real keyword counts; otherwise it writes a stub table with `NULL` value columns
so downstream Streamlit can still load the schema.

**Output:** `AIRBNB_INVESTMENT_DB.GOLD.REVIEW_THEMES`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
DB         = 'AIRBNB_INVESTMENT_DB'
SCHEMA_SIL = 'SILVER'
SCHEMA_GLD = 'GOLD'
TABLE_OUT  = 'REVIEW_THEMES'

PRICE_KEYWORDS       = 'cheap|expensive|value|price|cost|affordable|overpriced'
CLEANLINESS_KEYWORDS = 'clean|dirty|spotless|tidy|mess|dusty|filthy'
LOCATION_KEYWORDS    = 'location|central|transport|walk|nearby|tube|bus|station'
CHECKIN_KEYWORDS     = 'check-in|checkin|key|access|arrival|lockbox|host|welcome'

In [ ]:
def check_reviews_silver(session):
    result = session.sql('''
        SELECT COUNT(*) AS tbl_count
        FROM AIRBNB_INVESTMENT_DB.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = 'SILVER'
          AND TABLE_NAME = 'BRISTOL_REVIEWS_SILVER'
    ''').to_pandas()
    reviews_available = result['TBL_COUNT'][0] > 0
    print(f'BRISTOL_REVIEWS_SILVER available: {reviews_available}')
    return reviews_available


def build_stub(session):
    df = session.sql('''
        SELECT DISTINCT neighbourhood_cleansed
        FROM AIRBNB_INVESTMENT_DB.SILVER.BRISTOL_LISTINGS_SILVER
    ''').to_pandas()

    df['total_reviews']              = None
    df['mentions_price_count']       = None
    df['mentions_cleanliness_count'] = None
    df['mentions_location_count']    = None
    df['mentions_checkin_count']     = None
    df['pct_mentions_price']         = None
    df['pct_mentions_cleanliness']   = None
    df['pct_mentions_location']      = None
    df['pct_mentions_checkin']       = None
    df['avg_sentiment_score']        = None
    # TODO: Phase 2 — Snowflake Cortex SENTIMENT() once reviews confirmed
    df['top_theme']                  = None
    df['computed_at']                = pd.Timestamp.now()

    print(f'Built stub for {len(df)} neighbourhoods (BRISTOL_REVIEWS_SILVER not available)')
    return df


def build_themes(session):
    df = session.sql('''
        SELECT neighbourhood_cleansed, comments
        FROM AIRBNB_INVESTMENT_DB.SILVER.BRISTOL_REVIEWS_SILVER
        WHERE comments IS NOT NULL
    ''').to_pandas()

    df['comments'] = df['comments'].str.lower()

    df['mentions_price']       = df['comments'].str.contains(PRICE_KEYWORDS, regex=True, na=False)
    df['mentions_cleanliness'] = df['comments'].str.contains(CLEANLINESS_KEYWORDS, regex=True, na=False)
    df['mentions_location']    = df['comments'].str.contains(LOCATION_KEYWORDS, regex=True, na=False)
    df['mentions_checkin']     = df['comments'].str.contains(CHECKIN_KEYWORDS, regex=True, na=False)

    df_themes = df.groupby('neighbourhood_cleansed').agg(
        total_reviews              = ('comments', 'count'),
        mentions_price_count       = ('mentions_price', 'sum'),
        mentions_cleanliness_count = ('mentions_cleanliness', 'sum'),
        mentions_location_count    = ('mentions_location', 'sum'),
        mentions_checkin_count     = ('mentions_checkin', 'sum'),
    ).reset_index()

    for col in ['mentions_price_count', 'mentions_cleanliness_count',
                'mentions_location_count', 'mentions_checkin_count']:
        df_themes[col] = df_themes[col].astype(int)

    safe_total = df_themes['total_reviews'].replace(0, float('nan'))
    df_themes['pct_mentions_price']       = (df_themes['mentions_price_count']       / safe_total * 100).round(2)
    df_themes['pct_mentions_cleanliness'] = (df_themes['mentions_cleanliness_count'] / safe_total * 100).round(2)
    df_themes['pct_mentions_location']    = (df_themes['mentions_location_count']    / safe_total * 100).round(2)
    df_themes['pct_mentions_checkin']     = (df_themes['mentions_checkin_count']     / safe_total * 100).round(2)

    count_cols = [
        'mentions_price_count', 'mentions_cleanliness_count',
        'mentions_location_count', 'mentions_checkin_count',
    ]
    label_map = {
        'mentions_price_count':       'price',
        'mentions_cleanliness_count': 'cleanliness',
        'mentions_location_count':    'location',
        'mentions_checkin_count':     'checkin',
    }
    df_themes['top_theme'] = df_themes[count_cols].idxmax(axis=1).map(label_map)

    df_themes['avg_sentiment_score'] = None
    # TODO: Phase 2 — Snowflake Cortex SENTIMENT() per comment,
    # averaged per neighbourhood once reviews silver confirmed

    df_themes['computed_at'] = pd.Timestamp.now()

    print(f'Review themes computed for {len(df_themes)} neighbourhoods from {len(df)} comments')
    return df_themes


def write_gold(session, df):
    session.write_pandas(
        df,
        'REVIEW_THEMES',
        database='AIRBNB_INVESTMENT_DB',
        schema='GOLD',
        overwrite=True
    )
    print(f'Written {len(df)} rows to GOLD.REVIEW_THEMES')


def validate(session):
    df_val = session.sql('''
        SELECT
            COUNT(*) AS neighbourhood_count,
            COUNT(total_reviews) AS non_null_reviews,
            COUNT(top_theme) AS non_null_top_theme
        FROM AIRBNB_INVESTMENT_DB.GOLD.REVIEW_THEMES
    ''').to_pandas()
    print(df_val)
    if df_val['NON_NULL_REVIEWS'][0] == 0:
        print('Status: STUB — all value columns are NULL')
    else:
        print('Status: POPULATED — review themes computed')

In [ ]:
# --- Orchestration ---
reviews_available = check_reviews_silver(session)
if reviews_available:
    df = build_themes(session)
else:
    df = build_stub(session)
write_gold(session, df)
validate(session)

In [ ]:
df_val = session.sql('''
    SELECT
        COUNT(*) AS neighbourhood_count,
        COUNT(total_reviews) AS non_null_reviews,
        COUNT(top_theme) AS non_null_top_theme
    FROM AIRBNB_INVESTMENT_DB.GOLD.REVIEW_THEMES
''').to_pandas()
print(df_val)
if df_val['NON_NULL_REVIEWS'][0] == 0:
    print('Status: STUB — all value columns are NULL')
else:
    print('Status: POPULATED — review themes computed')

## Review Themes Complete

`GOLD.REVIEW_THEMES` is ready for Streamlit consumption.
Each row represents one Bristol neighbourhood.
Value columns are `NULL` (stub) until `BRISTOL_REVIEWS_SILVER` is confirmed and this notebook is re-run.